In [ ]:
# Cell 1: Setup and Installation (Run once, ~5 minutes)

import os
import sys

print("🔧 Installing dependencies...")

# Install core packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q gradio==3.50.2
!pip install -q imageio-ffmpeg
!pip install -q gfpgan basicsr realesrgan safetensors
!pip install -q pyngrok

# Check GPU
import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU")

# Clone SadTalker
if not os.path.exists('SadTalker'):
    print("\n📥 Cloning SadTalker...")
    !git clone https://github.com/OpenTalker/SadTalker.git
else:
    print("\n✅ SadTalker already cloned")

os.chdir('SadTalker')

# Download models
if not os.path.exists('checkpoints/SadTalker_V0.0.2_256.safetensors'):
    print("\n📥 Downloading models (~2GB)...")
    !bash scripts/download_models.sh
else:
    print("\n✅ Models already downloaded")

# Install SadTalker requirements
!pip install -q -r requirements.txt

print("\n" + "="*70)
print("✅ Setup complete! Ready to start API server.")
print("="*70)

In [ ]:
# Cell 2: Configure ngrok token

# Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ⚠️ REPLACE THIS!

if NGROK_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("⚠️  Please set your ngrok token above!")
    print("Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Ngrok token configured!")

In [ ]:
# Cell 3: Start SadTalker API Server with GPU

import sys
import os
os.chdir('/content/SadTalker')
sys.path.insert(0, '/content/SadTalker')

from src.gradio_demo import SadTalker
from pyngrok import ngrok
import gradio as gr
import torch

print("🚀 Initializing SadTalker with GPU...")

# Initialize SadTalker
sad_talker = SadTalker(
    checkpoint_path='checkpoints',
    config_path='src/config',
    lazy_load=True
)

print(f"✅ SadTalker initialized on {torch.cuda.get_device_name(0)}")

def generate_video(source_image, driven_audio, preprocess='crop', 
                  still_mode=False, expression_scale=1.0, size=256, enhancer=False):
    """Generate talking head video with GPU acceleration"""
    try:
        print(f"\n🎬 Generating video...")
        print(f"   Size: {size}x{size}")
        print(f"   Preprocess: {preprocess}")
        print(f"   Expression scale: {expression_scale}")
        
        result = sad_talker.test(
            source_image=source_image,
            driven_audio=driven_audio,
            preprocess=preprocess,
            still_mode=still_mode,
            expression_scale=expression_scale,
            enhancer='gfpgan' if enhancer else None,
            batch_size=2,
            size=size,
            pose_style=0
        )
        
        print(f"✅ Video generated: {result}")
        return result
        
    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        print(error_msg)
        return None

# Create Gradio interface
interface = gr.Interface(
    fn=generate_video,
    inputs=[
        gr.Image(type="filepath", label="Avatar Image"),
        gr.Audio(type="filepath", label="Audio (WAV or MP3)"),
        gr.Dropdown(['crop', 'resize', 'full'], value='crop', label="Preprocessing"),
        gr.Checkbox(value=False, label="Still Mode (less head movement)"),
        gr.Slider(0.0, 2.0, value=1.0, step=0.1, label="Expression Scale"),
        gr.Dropdown([256, 512], value=256, label="Video Size"),
        gr.Checkbox(value=False, label="Face Enhancer (slower)")
    ],
    outputs=gr.Video(label="Generated Video"),
    title="🎬 SadTalker GPU Server - Rafiki AI",
    description="Fast lip-sync video generation powered by Google Colab GPU",
    examples=[
        ["examples/source_image/full_body_1.png", "examples/driven_audio/bus_chinese.wav", "crop", False, 1.0, 256, False]
    ]
)

# Start with ngrok tunnel
public_url = ngrok.connect(7860)

print("\n" + "="*70)
print("🚀 SadTalker GPU API Server is RUNNING!")
print("="*70)
print(f"\n📡 Public URL: {public_url}")
print(f"\n⚠️  IMPORTANT: Copy this URL to your backend configuration!")
print(f"\nSet this in your backend:")
print(f"  export SADTALKER_API_URL='{public_url}'")
print(f"  export SADTALKER_MODE='api'")
print("\n" + "="*70)

# Launch Gradio
interface.launch(share=False, server_port=7860)

## Usage Instructions

Once the server is running:

1. **Copy the public URL** from the output above
2. **Update your backend** (`/home/subchief/5TECH/backend/config.py`):
   ```python
   SADTALKER_API_URL = "https://xxxx.ngrok.io"  # Your ngrok URL
   SADTALKER_MODE = "api"  # Use API mode
   ```
3. **Restart your backend** to use the GPU server

### Performance:
- **With GPU (T4):** 5-10 seconds per video
- **CPU only:** 2-10 minutes per video
- **50-100x speedup!** 🚀

### Notes:
- Keep this Colab tab open while using the API
- Free Colab sessions last up to 12 hours
- The ngrok URL changes each time you restart